# Graph-Paper AI - Colab Test

Vectorless Graph-RAG: PDF -> Graph -> Tree Search (LLM) -> Answer.

Works with or without a local Ollama instance.


## Setup


In [ ]:
# Clone the repo and install deps
REPO_URL = "https://github.com/ahmadoubg/graph-paper-ai.git"
REPO_DIR = "/content/graph-paper-ai"
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q pymupdf networkx httpx llama-cloud>=2.1


In [ ]:
from pathlib import Path
from pprint import pprint

import networkx as nx

from src.ingestion import build_section_tree, print_tree
from src.ingestion.tree import build_node_id_map
from src.llm.ollama_client import OllamaClient
from src.retrieval.tree_search import answer_query, tree_search


## 1. Get a PDF

Run this cell, then click the Upload button to select a PDF from your computer.


In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
PDF_PATH = Path(pdf_name)
OUTPUT_DIR = Path("output")
print(f"Using: {PDF_PATH} ({PDF_PATH.stat().st_size / 1024:.0f} KB)")


In [ ]:
from google.colab import userdata
key_api = userdata.get('LLAMACLOUD_API_KEY')


### Or download a sample paper automatically


In [ ]:
# Uncomment to skip upload and use a sample arXiv paper instead:
# PDF_URL = "https://arxiv.org/pdf/2401.12345.pdf"
# !wget -q -O paper.pdf {PDF_URL}
# PDF_PATH = Path("paper.pdf")
# OUTPUT_DIR = Path("output")
# print(f"Downloaded: {PDF_PATH}")


## 2. Parse with LlamaParse (items mode)

Uses expand=['items'] to get structured page elements: headings, text blocks, figures, tables.


In [ ]:
from llama_cloud import AsyncLlamaCloud
from google.colab import userdata
import json

async def parse_with_llamaparse(pdf_path):
    key_api = userdata.get('LLAMACLOUD_API_KEY')
    client = AsyncLlamaCloud(api_key=key_api)
    file_obj = await client.files.create(file=pdf_path, purpose="parse")
    result = await client.parsing.parse(
        file_id=file_obj.id,
        tier="agentic",
        expand=["items"],
        version='latest'
    )
    return result


In [ ]:
llamaparse_raw_result = await parse_with_llamaparse(PDF_PATH)
pages = llamaparse_raw_result.items.pages if llamaparse_raw_result else []
print(f"Pages: {len(pages)}")


In [ ]:
def build_complete_hierarchical_tree(raw_result):
    if not hasattr(raw_result, 'items') or not getattr(raw_result.items, 'pages', None):
        print("No pages found in response.")
        return []
    root_nodes = []
    node_counter = 0
    active_stack = []
    for idx, page in enumerate(raw_result.items.pages):
        page_num = idx + 1
        current_text_fragments = []
        current_heading = None
        current_level = None
        if not hasattr(page, 'items') or not page.items:
            continue
        for item in page.items:
            if item.type == 'heading':
                if current_heading and current_text_fragments:
                    text_summary = " ".join(current_text_fragments)
                    _add_text_to_last_node(active_stack, text_summary)
                    current_text_fragments = []
                current_heading = item.value if hasattr(item, 'value') else item.md.replace('#', '').strip()
                current_level = getattr(item, 'level', 1)
                node_id_str = f"{node_counter:04d}"
                node_counter += 1
                new_node = {
                    "node_id": node_id_str, "title": current_heading,
                    "page_index": page_num, "summary": "",
                    "visuals": [], "tables": [], "nodes": []
                }
                while active_stack and active_stack[-1]['level'] >= current_level:
                    active_stack.pop()
                if not active_stack:
                    root_nodes.append(new_node)
                else:
                    active_stack[-1]['node']['nodes'].append(new_node)
                active_stack.append({'level': current_level, 'node': new_node})
            elif item.type == 'text' and hasattr(item, 'value') and item.value:
                current_text_fragments.append(item.value)
            elif item.type in ['image', 'figure'] and active_stack:
                visual_text = getattr(item, 'md', '') or getattr(item, 'value', '')
                if visual_text:
                    current_text_fragments.append(visual_text)
                visual_info = {
                    "type": item.type,
                    "image_id": getattr(item, 'name', f"fig_{node_counter}"),
                    "page": page_num,
                    "path": getattr(item, 'path', None) or getattr(item, 'image_path', ''),
                    "bbox": [b.__dict__ for b in item.bbox] if getattr(item, 'bbox', None) else []
                }
                active_stack[-1]['node']['visuals'].append(visual_info)
            elif item.type == 'table' and active_stack:
                table_info = {
                    "page": page_num,
                    "markdown_content": getattr(item, 'md', '') or getattr(item, 'value', ''),
                    "bbox": [b.__dict__ for b in item.bbox] if getattr(item, 'bbox', None) else []
                }
                active_stack[-1]['node']['tables'].append(table_info)
        if current_text_fragments and active_stack:
            text_summary = " ".join(current_text_fragments)
            _add_text_to_last_node(active_stack, text_summary)
    return root_nodes

def _add_text_to_last_node(stack, text):
    last_node = stack[-1]['node']
    clean_text = text.strip()
    if last_node["summary"]:
        last_node["summary"] += " " + clean_text
    else:
        last_node["summary"] = clean_text

complete_tree = build_complete_hierarchical_tree(llamaparse_raw_result)
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total
print(f"Tree: {len(complete_tree)} top-level, {count_nodes(complete_tree)} total nodes")


In [ ]:
def display_document_structure(nodes, indent_level=0):
    if indent_level == 0:
        print("Document Structure:\n")
    for node in nodes:
        spacing = "  " * indent_level
        print(f"{spacing}[{node['node_id']}] {node['title']}  (p.{node['page_index']})")
        if "nodes" in node and node["nodes"]:
            display_document_structure(node["nodes"], indent_level + 1)

display_document_structure(complete_tree)


## 3. Build Graph from Tree

Convert the LlamaParse tree (summary/visuals/tables) into a project-compatible nx.DiGraph.


In [ ]:
def build_graph_from_tree(tree_nodes):
    graph = nx.DiGraph()
    def _add(parent, nodes):
        for node in nodes:
            nid = "tree_" + node["node_id"]
            sidx = int(node["node_id"])
            graph.add_node(nid, node_type="section", title=node["title"],
                content=node.get("summary", ""), page=node.get("page_index", 1), _section_idx=sidx)
            if parent:
                graph.add_edge(parent, nid, edge_type="contains")
            for v in node.get("visuals", []):
                vid = v.get("image_id", "fig_" + nid)
                graph.add_node(vid, node_type="figure", title=v.get("image_id", ""),
                    content=v.get("path", ""), page=v.get("page", node.get("page_index", 1)))
                graph.add_edge(nid, vid, edge_type="contains")
            for ti, t in enumerate(node.get("tables", [])):
                tid = "tbl_" + node["node_id"] + "_" + str(ti)
                graph.add_node(tid, node_type="text_block", title="",
                    content=t.get("markdown_content", ""), page=t.get("page", node.get("page_index", 1)))
                graph.add_edge(nid, tid, edge_type="contains")
            _add(nid, node.get("nodes", []))
    _add(None, tree_nodes)
    return graph


In [ ]:
graph = build_graph_from_tree(complete_tree)
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

section_nodes = build_section_tree(graph)
node_id_map = build_node_id_map(graph)

print("Top-level sections:", len(section_nodes))
print()
print(print_tree(section_nodes))
print()
if section_nodes:
    import json
    first = section_nodes[0]
    print(json.dumps({
        "title": first.title, "node_id": first.node_id,
        "page_index": first.page_index, "text": first.text[:500],
    }, indent=2))
print(f"node_id_map entries: {len(node_id_map)}")


## 4. Install Ollama (Colab)

Only needed to run with a real LLM. Skip if just inspecting the tree.


In [ ]:
import subprocess, time
!sudo apt update && sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh
process = subprocess.Popen(["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(7)
print("Ollama server started")


In [ ]:
import os
os.environ['PATH'] += os.pathsep + '/usr/local/bin'
!ollama pull qwen2.5vl:3b


In [ ]:
llm = OllamaClient(base_url="http://localhost:11434", model="qwen2.5vl:3b")
print(f"Ollama connected: {llm.is_available()}")


## 5. Query with Project Tree Search

Uses tree_search() (LLM picks [XXXX] IDs) and answer_query() (LLM synthesizes answer).


In [ ]:
QUERY = "What is the main contribution of this paper?"
search_result = tree_search(QUERY, graph, llm, node_id_map=node_id_map)
print(f"LLM selected: {search_result.selected_ids}")
print(f"Context: {len(search_result.context):,} chars")
print()
print("--- Context preview ---")
print(search_result.context[:1500])


In [ ]:
answer = answer_query(QUERY, search_result.context, llm)
print("Answer:")
print(answer)


In [ ]:
test_queries = [
    "What is gradient descent?",
    "Explain the cost function",
    "How does linear regression work?",
]
for q in test_queries:
    print()
    sr = tree_search(q, graph, llm, node_id_map=node_id_map)
    ans = answer_query(q, sr.context, llm)
    print(f"Q: {q}")
    print(f"A: {ans[:500]}...")
    print("-" * 40)
